# Módulo 5: Cálculo II
## Lección 6: Derivación de Campos Vectoriales

La derivación de campos vectoriales es la piedra angular del modelado matemático en la física clásica y la ingeniería moderna. Las leyes que gobiernan la dinámica de fluidos (Navier-Stokes), la electrodinámica clásica (ecuaciones de Maxwell) y la termodinámica de medios continuos requieren relacionar campos vectoriales espaciales con sus derivadas direccionales y matrices de transformación local.

En esta lección, estudiaremos formalmente la matriz Jacobiana y el concepto de diferenciabilidad en transformaciones vectoriales. Analizaremos las implicaciones geométricas y físicas de los teoremas de la función inversa y la función implícita, y resolveremos problemas de optimización con restricciones mediante los multiplicadores de Lagrange. Finalmente, nos adentraremos en el formalismo del cálculo tensorial básico en física a través del delta de Kronecker $\delta_{ij}$, el tensor alternante de Levi-Civita $\epsilon_{ijk}$ y la demostración automatizada de identidades vectoriales complejas con Python.

### Objetivos de Aprendizaje

Al finalizar esta lección, serás capaz de:
1. **Calcular la matriz Jacobiana** de transformaciones vectoriales de $\mathbb{R}^n$ a $\mathbb{R}^m$, analizando el cambio local inducido en coordenadas polares, cilíndricas y esféricas.
2. **Verificar la diferenciabilidad** de campos vectoriales mediante aproximaciones lineales locales.
3. **Aplicar la regla de la cadena vectorial** para componer transformaciones de variables múltiples.
4. **Evaluar la invertibilidad local** de transformaciones mediante el análisis del determinante Jacobiano en el Teorema de la Función Inversa.
5. **Resolver problemas de optimización con restricciones** aplicando los Multiplicadores de Lagrange, visualizando la tangencia de las curvas de nivel.
6. **Utilizar el delta de Kronecker $\delta_{ij}$ y el tensor de Levi-Civita $\epsilon_{ijk}$** para simplificar productos vectoriales y demostrar identidades vectoriales complejas en SymPy.

### 1. La Matriz Jacobiana y Diferenciabilidad

#### Matriz Jacobiana:
Sea $\mathbf{F}: \mathbb{R}^n \to \mathbb{R}^m$ una transformación vectorial dada por sus funciones componentes:

$$\mathbf{F}(x_1, x_2, \dots, x_n) = \left( F_1(x_1, \dots, x_n), \ F_2(x_1, \dots, x_n), \ \dots, \ F_m(x_1, \dots, x_n) \right)$$

Si las derivadas parciales de cada componente existen en un punto $\mathbf{a}$, la **matriz Jacobiana** $J_{\mathbf{F}}(\mathbf{a})$ es la matriz de tamaño $m \times n$ de derivadas parciales:

$$J_{\mathbf{F}}(\mathbf{a}) = \begin{pmatrix} \frac{\partial F_1}{\partial x_1} & \frac{\partial F_1}{\partial x_2} & \dots & \frac{\partial F_1}{\partial x_n} \\ \frac{\partial F_2}{\partial x_1} & \frac{\partial F_2}{\partial x_2} & \dots & \frac{\partial F_2}{\partial x_n} \\ \vdots & \vdots & \ddots & \vdots \\ \frac{\partial F_m}{\partial x_1} & \frac{\partial F_m}{\partial x_2} & \dots & \frac{\partial F_m}{\partial x_n} \end{pmatrix}_{\mathbf{x} = \mathbf{a}}$$

Esta matriz representa la mejor aproximación lineal (diferencial) de la transformación cerca del punto $\mathbf{a}$.

#### Diferenciabilidad Vectorial:
La transformación $\mathbf{F}$ es diferenciable en un punto $\mathbf{a} \in \mathbb{R}^n$ si existe una transformación lineal representada por la matriz $A$ (que coincide con la Jacobiana) tal que:

$$\lim_{\mathbf{h} \to \mathbf{0}} \frac{\|\mathbf{F}(\mathbf{a} + \mathbf{h}) - \mathbf{F}(\mathbf{a}) - J_{\mathbf{F}}(\mathbf{a})\mathbf{h}\|}{\|\mathbf{h}\|} = 0$$

#### Caso de Estudio: Coordenadas Polares
La transformación de coordenadas polares a cartesianas es:

$$\mathbf{G}(r, \theta) = \langle r \cos\theta, \ r \sin\theta \rangle$$

Su matriz Jacobiana es:

$$J_{\mathbf{G}}(r, \theta) = \begin{pmatrix} \cos\theta & -r \sin\theta \\ \sin\theta & r \cos\theta \end{pmatrix}$$

El determinante Jacobiano es $\det(J_{\mathbf{G}}) = r(\cos^2\theta + \sin^2\theta) = r$.

Visualizaremos cómo esta transformación deforma una rejilla polar en el plano cartesiano.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp
from mpl_toolkits.mplot3d import Axes3D

# Configuración de estilo visual
plt.style.use('seaborn-v0_8-whitegrid')

# Deformación de rejilla polar en el plano cartesiano
r_vals = np.linspace(0.5, 2.0, 5)
theta_vals = np.linspace(0, 2*np.pi, 100)

plt.figure(figsize=(6, 6))

# Círculos concéntricos (r constante)
for r in r_vals:
    x = r * np.cos(theta_vals)
    y = r * np.sin(theta_vals)
    plt.plot(x, y, 'b-', alpha=0.5)

# Rayos radiales (theta constante)
rayos = np.linspace(0, 2*np.pi, 8, endpoint=False)
r_rayo = np.linspace(0, 2.2, 10)
for th in rayos:
    x = r_rayo * np.cos(th)
    y = r_rayo * np.sin(th)
    plt.plot(x, y, 'r--', alpha=0.5)

plt.title('Transformación de Coordenadas Polares (Rejilla Deformada)')
plt.xlabel('Eje X')
plt.ylabel('Eje Y')
plt.axis('equal')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 2. Regla de la Cadena Vectorial

Si componemos dos transformaciones vectoriales diferenciables $\mathbf{w} = \mathbf{F}(\mathbf{y})$ y $\mathbf{y} = \mathbf{G}(\mathbf{x})$, la matriz Jacobiana de la composición $\mathbf{H} = \mathbf{F} \circ \mathbf{G}$ se calcula directamente multiplicando las matrices Jacobianas de cada transformación:

$$J_{\mathbf{F} \circ \mathbf{G}}(\mathbf{x}) = J_{\mathbf{F}}(\mathbf{G}(\mathbf{x})) \cdot J_{\mathbf{G}}(\mathbf{x})$$

Esta operación matricial simplifica la manipulación analítica de transformaciones complejas en cascada.

### 3. Teoremas de la Función Inversa y la Función Implícita

#### Teorema de la Función Inversa:
Sea $\mathbf{F}: \mathbb{R}^n \to \mathbb{R}^n$ una función diferenciable con derivadas parciales continuas en un entorno de $\mathbf{a}$. Si la matriz Jacobiana $J_{\mathbf{F}}(\mathbf{a})$ es invertible (es decir, su determinante Jacobiano es no nulo $\det(J_{\mathbf{F}}(\mathbf{a})) \neq 0$), entonces:
1. Existe un entorno abierto $U$ de $\mathbf{a}$ en el cual $\mathbf{F}$ es biyectiva y posee una inversa local $\mathbf{F}^{-1}$.
2. La función inversa es diferenciable y su matriz Jacobiana es la inversa de la Jacobiana original:
   
   $$J_{\mathbf{F}^{-1}}(\mathbf{F}(\mathbf{a})) = \left( J_{\mathbf{F}}(\mathbf{a}) \right)^{-1}$$

#### Teorema de la Función Implícita (Sistemas):
Sea $\mathbf{F}: \mathbb{R}^n \times \mathbb{R}^m \to \mathbb{R}^m$ una función diferenciable. Si en un punto $(\mathbf{a}, \mathbf{b})$ se cumple que $\mathbf{F}(\mathbf{a}, \mathbf{b}) = \mathbf{0}$ y la submatriz Jacobiana respecto a las variables dependientes $\mathbf{y}$:

$$\frac{\partial \mathbf{F}}{\partial \mathbf{y}}(\mathbf{a}, \mathbf{b})$$

es invertible, entonces existe una función implícita diferenciable $\mathbf{y} = \mathbf{g}(\mathbf{x})$ tal que $\mathbf{F}(\mathbf{x}, \mathbf{g}(\mathbf{x})) = \mathbf{0}$ en un entorno de $\mathbf{a}$.

### 4. Extremos Condicionados (Multiplicadores de Lagrange)

El método de los Multiplicadores de Lagrange permite hallar los valores extremos locales de una función objetivo $f(x, y)$ sujeta a una restricción de igualdad $g(x, y) = 0$.

#### Vínculo Geométrico:
En el punto de extremo condicionado, la curva de nivel de la función objetivo $f(x, y)$ debe ser tangente a la curva de la restricción $g(x, y) = 0$. Esto significa geométricamente que los vectores gradientes de ambas funciones son paralelos:

$$\nabla f(x, y) = \lambda \nabla g(x, y)$$

donde $\lambda$ es el **multiplicador de Lagrange**.

#### Caso de Estudio: Utilidad de Cobb-Douglas en Economía
Queremos maximizar la función de utilidad de Cobb-Douglas:

$$f(x, y) = x^{0.5} y^{0.5}$$

sujeta a la restricción presupuestaria lineal de precios $p_x = 2$, $p_y = 3$ y un presupuesto de $M = 12$:

$$g(x, y) = 2x + 3y - 12 = 0$$

Formamos la función Lagrangiana:

$$L(x, y, \lambda) = x^{0.5} y^{0.5} - \lambda(2x + 3y - 12)$$

Igualando las derivadas parciales a cero se obtiene el punto crítico óptimo condicionado en $(3, 2)$.

A continuación, graficaremos la superficie de utilidad en 3D, mostrando cómo la restricción presupuestaria corta a la superficie y cómo la tangencia ocurre en el punto óptimo.

In [ ]:
def f_utilidad(x, y):
    return np.sqrt(x * y)

x = np.linspace(0.1, 6.0, 100)
y = np.linspace(0.1, 6.0, 100)
X, Y = np.meshgrid(x, y)
Z = f_utilidad(X, Y)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, Z, cmap='terrain', alpha=0.5, edgecolor='none')

# Curva de restricción presupuestaria y = (12 - 2x)/3
rx = np.linspace(0.1, 5.9, 100)
ry = (12.0 - 2.0*rx) / 3.0
valid = ry >= 0
rz = f_utilidad(rx[valid], ry[valid])

ax.plot(rx[valid], ry[valid], rz, 'r-', linewidth=4.0, label='Restricción $2x + 3y = 12$')

# Punto óptimo (3, 2)
ax.scatter(3.0, 2.0, f_utilidad(3.0, 2.0), color='black', s=150, zorder=10, label='Punto Óptimo (3, 2)')

ax.set_title('Optimización Cobb-Douglas con Multiplicadores de Lagrange')
ax.set_xlabel('Bien X')
ax.set_ylabel('Bien Y')
ax.set_zlabel('Utilidad U')
ax.legend(loc='upper right', frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

### 5. Delta de Kronecker y Tensor de Levi-Civita

En física teórica y cálculo tensorial, se utilizan dos tensores de componentes constantes para expresar de forma compacta operaciones algebraicas:

- **Delta de Kronecker ($\delta_{ij}$):** Actúa como un selector de componentes (la matriz identidad):
  
  $$\delta_{ij} = \begin{cases} 1 & \text{si } i = j \\ 0 & \text{si } i \neq j \end{cases}$$
  
- **Tensor alternante de Levi-Civita ($\epsilon_{ijk}$):** Es el tensor totalmente antisimétrico en tres dimensiones:
  
  $$\epsilon_{ijk} = \begin{cases} +1 & \text{si } (i, j, k) \text{ es una permutación par de } (1, 2, 3) \\ -1 & \text{si } (i, j, k) \text{ es una permutación impar de } (1, 2, 3) \\ 0 & \text{si hay dos o más índices repetidos} \end{cases}$$

#### Aplicación al Producto Vectorial:
El producto vectorial $\mathbf{w} = \mathbf{u} \times \mathbf{v}$ se expresa formalmente en notación de índices como:

$$w_i = (\mathbf{u} \times \mathbf{v})_i = \sum_{j,k} \epsilon_{ijk} u_j v_k$$

### 6. Operadores Diferenciales e Identidades Vectoriales

En el análisis de campos vectoriales en física, definimos tres operadores diferenciales espaciales de primer orden utilizando el operador del (nablas):

- **Gradiente:** $\nabla f = \langle f_x, f_y, f_z \rangle$ (asigna un campo vectorial a un campo escalar).
- **Divergencia:** $\nabla \cdot \mathbf{F} = \frac{\partial F_1}{\partial x} + \frac{\partial F_2}{\partial y} + \frac{\partial F_3}{\partial z}$ (asigna un campo escalar a un campo vectorial).
- **Rotacional:** $\nabla \times \mathbf{F} = \langle F_{3,y} - F_{2,z}, \ F_{1,z} - F_{3,x}, \ F_{2,x} - F_{1,y} \rangle$ (asigna un campo vectorial a otro campo vectorial).

#### Identidad del Rotacional del Rotacional:
Una de las identidades más utilizadas en la teoría electromagnética de Maxwell es:

$$\nabla \times (\nabla \times \mathbf{A}) = \nabla(\nabla \cdot \mathbf{A}) - \nabla^2 \mathbf{A}$$

donde $\nabla^2 \mathbf{A}$ es el Laplaciano vectorial (el operador Laplaciano aplicado a cada componente de $\mathbf{A}$).

Utilizaremos `SymPy` para verificar simbólicamente la validez de esta identidad para un campo vectorial genérico de tres dimensiones $\mathbf{A}(x, y, z) = \langle P(x,y,z), Q(x,y,z), R(x,y,z) \rangle$.

In [ ]:
x_sym, y_sym, z_sym = sp.symbols('x y z', real=True)

# Campos componentes genéricos
P = sp.Function('P')(x_sym, y_sym, z_sym)
Q = sp.Function('Q')(x_sym, y_sym, z_sym)
R = sp.Function('R')(x_sym, y_sym, z_sym)
A_sym = sp.Matrix([P, Q, R])

# Definición de operadores diferenciales didácticos
def rot(F):
    return sp.Matrix([
        F[2].diff(y_sym) - F[1].diff(z_sym),
        F[0].diff(z_sym) - F[2].diff(x_sym),
        F[1].diff(x_sym) - F[0].diff(y_sym)
    ])

def div(F):
    return F[0].diff(x_sym) + F[1].diff(y_sym) + F[2].diff(z_sym)

def grad(f):
    return sp.Matrix([f.diff(x_sym), f.diff(y_sym), f.diff(z_sym)])

def laplaciano_vectorial(F):
    return sp.Matrix([
        F[0].diff(x_sym, 2) + F[0].diff(y_sym, 2) + F[0].diff(z_sym, 2),
        F[1].diff(x_sym, 2) + F[1].diff(y_sym, 2) + F[1].diff(z_sym, 2),
        F[2].diff(x_sym, 2) + F[2].diff(y_sym, 2) + F[2].diff(z_sym, 2)
    ])

# Lado izquierdo: rot(rot(A))
lado_izq = sp.simplify(rot(rot(A_sym)))

# Lado derecho: grad(div(A)) - lap(A)
lado_der = sp.simplify(grad(div(A_sym)) - laplaciano_vectorial(A_sym))

diferencia = sp.simplify(lado_izq - lado_der)
print("Diferencia Simbólica calculada (debe ser el vector cero [0, 0, 0]^T):")
sp.pprint(diferencia)

assert diferencia == sp.Matrix([0, 0, 0]), "La identidad vectorial del rotacional no es válida."
print("\n¡Identidad del rotacional del rotacional demostrada y verificada con éxito!")

### Resumen de Conceptos Clave

1. **Matriz Jacobiana:** Matriz de derivadas parciales que representa la mejor aproximación lineal de una transformación vectorial en un punto.
2. **Teoremas de Inversión e Implícita:** Establecen condiciones algebraicas locales (invertibilidad de la Jacobiana) para la existencia de funciones inversas e implícitas en varias variables.
3. **Multiplicadores de Lagrange:** Método geométrico para optimizar funciones con restricciones mediante la tangencia de las curvas de nivel.
4. **Álgebra Tensorial y Nabla:** El delta de Kronecker $\delta_{ij}$, el tensor de Levi-Civita $\epsilon_{ijk}$ y los operadores diferenciales gradiente, divergencia y rotacional unifican la física matemática de campos vectoriales.

### Referencias Bibliográficas

- Boyce, W. E., & DiPrima, R. C. (2012). *Elementary Differential Equations and Boundary Value Problems*. John Wiley & Sons.
- Kreyszig, E. (2011). *Advanced Engineering Mathematics* (10th ed.). John Wiley & Sons.
- Stewart, J. (2012). *Calculus: Early Transcendentals* (7th ed.). Cengage Learning.
- Arfken, G. B., & Weber, H. J. (2005). *Mathematical Methods for Physicists* (6th ed.). Academic Press.